# FACULTY SOLUTION — Week 1: Inheriting the seat
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['benchmarks', 'upstream_assets', 'refining_assets', 'rotterdam_cost_split', 'book_values', 'product_yields', 'worked_example_assets']}

In [ ]:

def compute(d):
    b = {x.parameter: x.value for x in d['benchmarks'].itertuples()}
    up = d['upstream_assets'].set_index('asset'); rf = d['refining_assets'].set_index('refinery')
    rs = {x.parameter: x.value for x in d['rotterdam_cost_split'].itertuples()}
    def real(a): return b['wti' if up.loc[a, 'benchmark'] == 'WTI' else 'brent'] + up.loc[a, 'differential']
    r = {}
    r['permian_realized'] = real('Permian')
    r['permian_margin'] = r['permian_realized'] - up.loc['Permian', 'lifting'] - up.loc['Permian', 'logistics']
    r['norway_pretax'] = real('Norwegian') - up.loc['Norwegian', 'lifting'] - up.loc['Norwegian', 'logistics']
    r['norway_posttax'] = r['norway_pretax'] * (1 - up.loc['Norwegian', 'fiscal_rate'])
    r['norway_2usd_loss_posttax'] = -2.0 * (1 - up.loc['Norwegian', 'fiscal_rate'])
    r['kessana_company'] = (real('Kessana') - up.loc['Kessana', 'lifting'] - up.loc['Kessana', 'logistics']) * (1 - up.loc['Kessana', 'fiscal_rate'])
    r['br_net'] = rf.loc['Baton Rouge', 'crack'] + rf.loc['Baton Rouge', 'complexity'] - rf.loc['Baton Rouge', 'opex']
    r['rot_contribution'] = rf.loc['Rotterdam', 'crack'] + rf.loc['Rotterdam', 'complexity'] - rs['variable_cost']
    r['rot_net'] = r['rot_contribution'] - rs['fixed_cost']
    r['rot_shutdown_crack'] = rs['variable_cost'] - rf.loc['Rotterdam', 'complexity']
    r['rot_current_crack'] = rf.loc['Rotterdam', 'crack']
    r['sing_halden_share'] = (rf.loc['Singapore', 'crack'] + rf.loc['Singapore', 'complexity'] - rf.loc['Singapore', 'opex']) * rf.loc['Singapore', 'halden_share']
    econ = {'Permian': r['permian_margin'], 'Norwegian': r['norway_posttax'], 'Kessana': r['kessana_company'],
            'Baton Rouge': r['br_net'], 'Rotterdam': r['rot_net'], 'Singapore': r['sing_halden_share']}
    rep = d['book_values'].set_index('asset')['reported_profit_musd'].to_dict()
    r['economic_rank'] = ' > '.join(sorted(econ, key=lambda k: -econ[k]))
    r['reported_rank'] = ' > '.join(sorted(rep, key=lambda k: -rep[k]))
    r['top_economic_asset'] = max(econ, key=econ.get)
    return r


In [ ]:
fx = json.load(open('../fixtures/week1_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')